# EIS-TIDY3D-LEAN-001 — SOI directional-coupler FDTD

This notebook consumes the JSON emitted by the Lean4 executable and builds a Tidy3D FDTD simulation of a two-waveguide SOI directional coupler. The initial parameter set is explicitly illustrative and assumed—not foundry-qualified or measured.

**Cloud execution is opt-in.** The notebook builds and inspects the simulation without submitting it. Set `RUN_CLOUD = True` only after reviewing the geometry, data classification, FlexCompute workspace and expected credit use.


In [ ]:
# In the repository root, install the bridge and Tidy3D dependencies first.
%pip install -r requirements-tidy3d.txt


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
from tidy3d_bridge.directional_coupler import load_spec, build_directional_coupler, launch_cloud

ROOT = Path.cwd()
SPEC_PATH = ROOT / 'examples/tidy3d/eis_soi_dc_spec.json'
spec = load_spec(SPEC_PATH)
print(spec['spec_id'], spec['evidence_class'], spec['process_status'])


## 1. Build the FDTD model

The input mode plane is restricted to the lower waveguide so that the mode source does not unintentionally launch a supermode spanning both arms. Two output mode monitors separately measure the through and cross ports.


In [ ]:
sim, metadata = build_directional_coupler(spec)
print(metadata)
print('FDTD grid cells:', sim.grid.num_cells)
sim.plot(z=0)
plt.show()


## 2. Explicit cloud-submission gate

No job is submitted unless this cell is deliberately changed. Review FlexCompute account credits and simulation settings before enabling it.


In [ ]:
RUN_CLOUD = False  # Change to True only after explicit review and authorization.
if RUN_CLOUD:
    sim_data = launch_cloud(
        sim,
        task_name='EIS-TIDY3D-LEAN-001-SOI-DC',
        path='data/eis_soi_directional_coupler.hdf5',
    )
else:
    print('Cloud submission skipped. No FlexCompute job has been launched.')


## 3. Inspect modal output (after a completed cloud run)

The monitor amplitudes are complex modal coefficients. Their squared magnitudes provide modal power proxies under the Tidy3D mode normalization. Keep the raw complex amplitudes: EIS needs phase as well as amplitude to calculate the complex transfer and `κ_A` in subsequent work. Confirm the exact normalization convention before comparing absolute power across separate runs.


In [ ]:
# Run only after RUN_CLOUD=True and a successful simulation.
if 'sim_data' in globals():
    through = sim_data['through'].amps.sel(direction='+').isel(mode_index=0)
    cross = sim_data['cross'].amps.sel(direction='+').isel(mode_index=0)
    p_through = np.abs(through.values) ** 2
    p_cross = np.abs(cross.values) ** 2
    denom = p_through + p_cross
    cross_fraction = np.divide(p_cross, denom, out=np.full_like(p_cross, np.nan), where=denom > 0)
    wavelengths_um = td.C_0 / through.coords['f'].values
    plt.plot(wavelengths_um, cross_fraction, label='cross-port fraction')
    plt.xlabel('Wavelength (µm)')
    plt.ylabel('Cross-port fraction')
    plt.grid(True)
    plt.legend()
    plt.show()
    print('Retain complex through/cross amplitudes for phase and κA analysis.')
else:
    print('No simulation data available yet.')


## 4. Evidence interpretation and next gate

This first simulation can establish a numerical wavelength-dependent complex transfer for the defined geometry. It does **not** establish a measured device response, a foundry process model, fabrication tolerance, thermal/electrical drive behavior, or KKA system performance.

The next gate is to export complex through/cross amplitudes with solver version, exact Lean spec hash, task ID, mesh settings and monitor normalization. Python independently calculates the transfer and uncertainty; Lean4 accepts only explicitly stated mathematical predicates. Do not collapse amplitude-only power ratios into the EIS complex-field contract.
